In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from keras.applications import InceptionV3
from keras.applications.inception_v3 import preprocess_input
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import Adam, SGD
from tensorflow.keras.preprocessing import image
import numpy as np
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.applications import ResNet50
from keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
##Densenet121
from keras.applications import DenseNet121
from keras.applications.densenet import preprocess_input

In [ ]:
IM_WIDTH, IM_HEIGHT = 224, 224
NB_EPOCHS = 4
BAT_SIZE = 32
FC_SIZE = 128
NB_DENSE_LAYERS_TO_FREEZE = 169  # Adjust as needed
dataset_folder = "C:\\Users\\Sandusha\\FYP\\Dataset"
train_dir = "Dataset/Train"
val_dir = "Dataset/Val"
test_dir = "Dataset/Test"

def add_new_last_layer_densenet(base_model, nb_classes):
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(FC_SIZE, activation='relu')(x)
    predictions = Dense(nb_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    return model

def setup_to_transfer_learn_densenet(model, base_model):
    for layer in base_model.layers:
        layer.trainable = False
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

def setup_to_finetune_densenet(model):
    for layer in model.layers[:NB_DENSE_LAYERS_TO_FREEZE]:
        layer.trainable = False
    for layer in model.layers[NB_DENSE_LAYERS_TO_FREEZE:]:
        layer.trainable = True
    model.compile(optimizer=SGD(learning_rate=0.0001, momentum=0.9), loss='categorical_crossentropy', metrics=['accuracy'])

def train_densenet(train_dir, val_dir, nb_epoch=NB_EPOCHS, batch_size=BAT_SIZE, output_model_file="densenet121-ft.model", plot=True):
    nb_train_samples = get_nb_files(train_dir)
    nb_classes = len(glob.glob(train_dir + "/*"))
    nb_val_samples = get_nb_files(val_dir)

    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        fill_mode='nearest',  # Filling strategy for rotated images
    )

    val_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        fill_mode='nearest',
    )
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IM_WIDTH, IM_HEIGHT),
        batch_size=batch_size,
        class_mode='categorical'
    )

    validation_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(IM_WIDTH, IM_HEIGHT),
        batch_size=batch_size,
        class_mode='categorical'
    )

    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(IM_WIDTH, IM_HEIGHT, 3))
    model = add_new_last_layer_densenet(base_model, nb_classes)

    setup_to_transfer_learn_densenet(model, base_model)

    history_tl = model.fit(
        train_generator,
        epochs=nb_epoch,
        steps_per_epoch=nb_train_samples // batch_size,
        validation_data=validation_generator,
        validation_steps=nb_val_samples // batch_size,
    )

    setup_to_finetune_densenet(model)

    history_ft = model.fit(
        train_generator,
        steps_per_epoch=nb_train_samples // batch_size,
        epochs=nb_epoch,
        validation_data=validation_generator,
        validation_steps=nb_val_samples // batch_size,
    )

    model.save(output_model_file)

    if plot:
        plot_training(history_ft)

    return model

# Train the DenseNet121 model and get the trained model
trained_densenet_model = train_densenet(train_dir=train_dir, val_dir=val_dir, nb_epoch=NB_EPOCHS, batch_size=BAT_SIZE, output_model_file="densenet121-ft.model", plot=True)

# Display the model summary
trained_densenet_model.summary()